# AI Powered (Gesture/Voice) Hybrid Transportation Prototye

In [11]:
import cv2
import mediapipe as mp
import pyttsx3
import time
import os
import queue
import json
import sounddevice as sd
from vosk import Model, KaldiRecognizer
import threading

### Simulation Mode

In [12]:
SIMULATE = True

if not SIMULATE:
    import serial
    try:
        bluetooth = serial.Serial('COM5', 9600)
        time.sleep(2)
        print("✅ Bluetooth connected.")
    except Exception as e:
        print("❌ Bluetooth Error:", e)
        exit()

### Voice Control

***Speak function (Vosk model) :***

In [13]:
engine = pyttsx3.init()
speech_queue = queue.Queue()

def speak_worker():
    while True:
        text = speech_queue.get()
        if text is None:
            break
        print("Bot:", text)
        engine.say(text)
        engine.runAndWait()
        speech_queue.task_done()

speech_thread = threading.Thread(target=speak_worker, daemon=True)
speech_thread.start()

def speak(text):
    speech_queue.put(text)

Exception in thread Thread-4:
Traceback (most recent call last):
  File "C:\Users\MOHAMED ATHIS\anaconda3\envs\gesturebot\lib\threading.py", line 980, in _bootstrap_inner
    self.run()
  File "C:\Users\MOHAMED ATHIS\anaconda3\envs\gesturebot\lib\site-packages\ipykernel\ipkernel.py", line 766, in run_closure
    _threading_Thread_run(self)
  File "C:\Users\MOHAMED ATHIS\anaconda3\envs\gesturebot\lib\threading.py", line 917, in run
    self._target(*self._args, **self._kwargs)
  File "C:\Users\MOHAMED ATHIS\AppData\Local\Temp\ipykernel_15800\1631530050.py", line 11, in speak_worker
  File "C:\Users\MOHAMED ATHIS\anaconda3\envs\gesturebot\lib\site-packages\pyttsx3\engine.py", line 180, in runAndWait
    raise RuntimeError('run loop already started')
RuntimeError: run loop already started


Bot: Voice control activated. Say forward, stop, left, right, or exit.


***Listener :***

In [14]:
def listen_command():
    try:
        q = queue.Queue()

        def callback(indata, frames, time, status):
            if status:
                print("⚠️", status)
            q.put(bytes(indata))

        model_path = "vosk-model-small-en-us-0.15"  # Make sure folder is in your project dir
        if not os.path.exists(model_path):
            speak("Vosk model not found.")
            return None

        model = Model(model_path)
        recognizer = KaldiRecognizer(model, 16000)

        with sd.RawInputStream(samplerate=16000, blocksize=8000, dtype='int16',
                               channels=1, callback=callback):
            print("🎤 Listening (Vosk)...")
            speak("Listening...")
            result_text = ""

            timeout_counter = 0
            while True:
                if not q.empty():
                    data = q.get()
                    if recognizer.AcceptWaveform(data):
                        result = json.loads(recognizer.Result())
                        result_text = result.get("text", "")
                        break
                else:
                    timeout_counter += 1
                    time.sleep(0.1)
                    if timeout_counter > 100:  # ~10 sec timeout
                        speak("You didn’t say anything.")
                        return None

        print("You said:", result_text)
        return result_text.lower()

    except Exception as e:
        print("❌ Error:", e)
        speak("Could not recognize your voice.")
        return None

***Voice Command setup:***

In [15]:
def voice_control():
    speak("Voice control activated. Say forward, stop, left, right, or exit.")
    while True:
        command = listen_command()
        if command:
            if "forward" in command:
                send_command('F')
                speak("Moving forward")
            elif "stop" in command:
                send_command('S')
                speak("Stopping")
            elif "left" in command:
                send_command('L')
                speak("Turning left")
            elif "right" in command:
                send_command('R')
                speak("Turning right")
            elif "exit" in command:
                speak("Exiting voice control.")
                break
            else:
                speak("Unknown command")

### Gesture Control

***Gesture Commands:***

In [16]:
def detect_gesture(landmarks):
    finger_tips = [8, 12, 16, 20]  # Index, Middle, Ring, Pinky
    fingers_up = []
    for tip in finger_tips:
        tip_y = landmarks.landmark[tip].y
        pip_y = landmarks.landmark[tip - 2].y
        fingers_up.append(tip_y < pip_y)

    # Map gestures
    if fingers_up == [True, True, True, True]:
        return "Forward"
    elif fingers_up == [False, False, False, False]:
        return "Stop"
    elif fingers_up == [True, True, False, False]:
        return "Right"
    elif fingers_up == [True, False, False, False]:
        return "Left"
    elif fingers_up == [True, False, False, True]:  # Thumb + Pinky
        return "Exit"
    else:
        return "Other"

***Gesture Function:***

In [17]:
def gesture_control():
    speak("Gesture control activated. Use your hand to control.")
    mp_hands = mp.solutions.hands
    hands = mp_hands.Hands(max_num_hands=1, min_detection_confidence=0.7)
    mp_draw = mp.solutions.drawing_utils
    cap = cv2.VideoCapture(0)

    last_gesture = ""
    bot_status = ""

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        frame = cv2.flip(frame, 1)
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = hands.process(rgb)

        if results.multi_hand_landmarks:
            for hand_landmarks in results.multi_hand_landmarks:
                mp_draw.draw_landmarks(frame, hand_landmarks, mp_hands.HAND_CONNECTIONS)
                gesture = detect_gesture(hand_landmarks)

                if gesture != last_gesture:
                    last_gesture = gesture
                    if gesture == "Forward":
                        send_command('F')
                        bot_status = "Moving Forward"
                    elif gesture == "Stop":
                        send_command('S')
                        bot_status = "Stopping"
                    elif gesture == "Left":
                        send_command('L')
                        bot_status = "Turning Left"
                    elif gesture == "Right":
                        send_command('R')
                        bot_status = "Turning Right"
                    elif gesture == "Exit":
                        speak("Exit gesture detected. Closing gesture control.")
                        cap.release()
                        cv2.destroyAllWindows()
                        return
                    else:
                        bot_status = "Unrecognized"

                # Display Gesture & Bot Status
                cv2.putText(frame, f'Gesture: {gesture}', (10, 30),
                            cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
                cv2.putText(frame, f'Bot: {bot_status}', (10, 70),
                            cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2)

        cv2.imshow("Gesture Control", frame)
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break

    cap.release()
    cv2.destroyAllWindows()


### Output Type

In [18]:
def send_command(command):
    if SIMULATE:
        print(f"[SIMULATED] Command sent: {command}")
    else:
        bluetooth.write(command.encode())
        print(f"[BT] Sent: {command}")


## Main

In [ ]:
def main():
    print("=== AI Robot Assistant ===")
    print("1. Voice Control 🎤")
    print("2. Gesture Control ✋")
    mode = input("Enter choice: ")

    if mode == '1':
        voice_control()
    elif mode == '2':
        gesture_control()
    else:
        print("Invalid choice.")

if __name__ == "__main__":
    main()

=== AI Robot Assistant ===
1. Voice Control 🎤
2. Gesture Control ✋


Enter choice:  1


🎤 Listening (Vosk)...
You said: i would enable it and i commented on learning little need a car
🎤 Listening (Vosk)...
You said: faster than you might have a killer listen what other was can visit
🎤 Listening (Vosk)...
You said: that would i'd learn how to live live within they do not let me know when i know that either
🎤 Listening (Vosk)...
